<a href="https://colab.research.google.com/github/AgatiDoc/machine-learning-for-image-analysis/blob/main/Kopia_notatnika_assignment_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Segmentation with U-Net

In the previous assignments you built classifiers that assign a single label to an entire image. This assignment tackles a harder problem: **semantic segmentation** assigning a class label to *every pixel*. We will also go one step further with **instance segmentation**, which distinguishes individual objects of the same class.

We work with fluorescence microscopy images of cell nuclei from the [Kaggle Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018/) challenge. Our goals are:

1. Train a **U-Net** to classify each pixel as *background* (0), *nucleus interior* (1), or *nucleus boundary* (2) the **three-class** segmentation task.
2. Use the pretrained **CellPose** model to perform instance segmentation and count individual nuclei.

---

## Grading

**Do not rename any function.** The autograder uses exact name matching.

| Task | Function | Points |
|:-----|:---------|:------:|
| 1 | `nearest_neighbor_upsampling` | 10 |
| 2 | `transpose_conv` | 10 |
| 3 | `iou` | 10 |
| 4 | `compute_instance_stats` | 10 |
| | **Total** | **40** |

---

## GPU Setup

Training requires a GPU. In Google Colab go to **Runtime → Change runtime type → GPU** before running any cell. Without a GPU, training will be very slow.


In [ ]:
# NOTE: Execute this cell first. It installs all required packages.
!pip install -q "numpy<2.1" cellpose zarr albumentations mahotas

# Download additonal utility files and data
import os

if not os.path.exists("data"):
    !git clone https://github.com/hpi-mlia-2026/coding_assignment_04.git
    %cd coding_assignment_04

In [ ]:
%matplotlib inline

import glob
import os
import zipfile

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import zarr
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if str(device) == "cpu":
    print("WARNING: No GPU found. Enable GPU in Colab (Runtime > Change runtime type > GPU).")
else:
    print("GPU detected. You are ready to train!")

In [ ]:
# Extract the dataset (only needed once)
if not os.path.exists("dsb2018"):
    print("Extracting data.zip...")
    with zipfile.ZipFile("data.zip", "r") as zf:
        zf.extractall(".")
    print("Done.")

train_files = sorted(glob.glob(os.path.join("dsb2018", "train", "*.zarr")))
val_files   = sorted(glob.glob(os.path.join("dsb2018", "val",   "*.zarr")))
test_files  = sorted(glob.glob(os.path.join("dsb2018", "test",  "*.zarr")))
print(f"Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)} images")

---

# Part 1 — Dataset Inspection

The dataset contains grayscale fluorescence microscopy images of cell nuclei. Each image is 360×360 pixels. The ground truth labels use three classes:

| Class | Value | Meaning |
|:------|:-----:|:--------|
| Background | 0 | No nucleus |
| Interior | 1 | Inside a nucleus |
| Boundary | 2 | Border between adjacent nuclei |

Separating interior from boundary is the key challenge: it allows the model to tell touching nuclei apart, which plain foreground/background segmentation cannot do.

Images are stored in [Zarr](https://zarr.dev/) format, an efficient format for large N-dimensional arrays.


In [ ]:
class NucleusDataset(Dataset):
    """Kaggle DSB-2018 nucleus dataset with optional augmentation."""

    # Label colours for visualisation: black / green / white
    LABEL_COLORS = np.array([[0, 0, 0], [0, 100, 0], [255, 255, 255]], dtype=np.uint8)

    def __init__(self, files, augment=False):
        self.files = files
        if augment:
            self.aug = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Rotate(limit=45, border_mode=0, p=0.7),
            ])
        else:
            self.aug = None

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = zarr.open(self.files[idx])
        raw   = np.array(data["volumes/raw"],          dtype=np.float32)   # (1, H, W)
        label = np.array(data["volumes/gt_threeclass"], dtype=np.int64)    # (1, H, W)

        # Zero-mean, unit-variance normalisation per image
        raw = (raw - raw.mean()) / (raw.std() + 1e-8)

        if self.aug is not None:
            raw_hwc   = raw.transpose(1, 2, 0)                         # HWC for albumentations
            label_hwc = label.transpose(1, 2, 0).astype(np.float32)
            out       = self.aug(image=raw_hwc, mask=label_hwc)
            raw       = out["image"].transpose(2, 0, 1).astype(np.float32)
            label     = out["mask"].transpose(2, 0, 1).astype(np.int64)

        return (
            torch.tensor(raw,   dtype=torch.float32),
            torch.tensor(label, dtype=torch.long),
        )

    @classmethod
    def label_to_rgb(cls, label_2d):
        """Convert a (H, W) integer label map to an (H, W, 3) RGB image."""
        return cls.LABEL_COLORS[label_2d.clip(0, 2)]


# Quick check
temp_dataset = NucleusDataset(train_files)
img, lbl = temp_dataset[0]
print(f"Image shape: {tuple(img.shape)}  Label shape: {tuple(lbl.shape)}")
print(f"Unique label values: {torch.unique(lbl).tolist()}")

In [ ]:
# Visualise three random training examples with their three-class labels
rng = np.random.default_rng(42)
indices = rng.choice(len(temp_dataset), size=3, replace=False)

fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for row, idx in enumerate(indices):
    img_t, lbl_t = temp_dataset[int(idx)]
    img_np = img_t.squeeze().numpy()
    lbl_np = lbl_t.squeeze().numpy().astype(np.int64)

    axes[row, 0].imshow(img_np, cmap="gray")
    axes[row, 0].set_title(f"Raw image #{idx}")
    axes[row, 0].axis("off")

    lbl_rgb = NucleusDataset.label_to_rgb(lbl_np)
    axes[row, 1].imshow(lbl_rgb)
    axes[row, 1].set_title("Ground truth segmentation mask")
    axes[row, 1].axis("off")

plt.suptitle("Three-class ground truth labels\n(black=BG, green=interior, white=boundary)\n", fontsize=13)
plt.tight_layout()
plt.show()

---

# Part 2 — Task 1: Nearest Neighbor Upsampling
<a id="task-1"></a>

## Background: Upsampling in the U-Net Decoder

The encoder of a U-Net repeatedly halves the spatial resolution using MaxPooling. The decoder has to reverse this process, increasing resolution step by step back to the original size.

The simplest upsampling strategy is **nearest neighbor (or constant) upsampling**: each input pixel is replicated into an `upsampling_factor × upsampling_factor` block of output pixels. For example, with `upsampling_factor=2` a 3×3 feature map becomes a 6×6 feature map where every value appears in a 2×2 block. No learnable parameters are involved.

## Task

Implement `nearest_neighbor_upsampling` as a pure NumPy function for a single-channel 2D feature map. The parameter `upsampling_factor` simultaneously acts as the stride and the block size — each input pixel is repeated `upsampling_factor` times in both spatial dimensions.


In [ ]:
def nearest_neighbor_upsampling(x: np.ndarray, upsampling_factor: int) -> np.ndarray:
    """Nearest-neighbor (constant) upsampling for a single-channel 2D feature map.

    Parameters
    ----------
    x : np.ndarray, shape (H_in, W_in)
        Input feature map.
    upsampling_factor : int
        Each spatial dimension is scaled by this factor. Acts as both the
        stride and the block size.

    Returns
    -------
    out : np.ndarray, shape (H_in * upsampling_factor, W_in * upsampling_factor)
        Upsampled feature map.
    """
    out = np.repeat(x, upsampling_factor, axis=0)   # replicate each row `factor` times
    out = np.repeat(out, upsampling_factor, axis=1)  # replicate each column `factor` times

    return out


In [ ]:
# ── Task 1 Self-Test ───────────────────────────────────────────────────
# Upsampling a 2×2 array with factor=2 should produce a 4×4 array
# where each value is repeated in a 2×2 block.
x_test = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
expected = np.array([
    [1., 1., 2., 2.],
    [1., 1., 2., 2.],
    [3., 3., 4., 4.],
    [3., 3., 4., 4.],
], dtype=np.float32)
got = nearest_neighbor_upsampling(x_test, upsampling_factor=2)
print("Expected output:")
print(expected)
print("\nYour output:")
print(got)
assert np.allclose(got, expected), \
    "Output does not match — check your implementation."
print("\nTask 1 self-test passed!")


---

# Part 3 — Task 2: Transposed Convolution
<a id="task-2"></a>

## Background: Learnable Upsampling in the U-Net Decoder

Nearest-neighbor upsampling simply copies pixel values without any learnable parameters. A more powerful alternative is the **transposed convolution** (sometimes called deconvolution or up-convolution), which is the operation used in `nn.ConvTranspose2d`.

Instead of sliding a kernel over the input and *collecting* values into each output position, you iterate over each input position and *scatter* a scaled copy of the kernel into the corresponding region of the output. For simplicity, we set **stride = kernel size**, so adjacent output blocks do not overlap.

The output size follows from the input size and the kernel size:

$$H_{\text{out}} = (H_{\text{in}} - 1) \cdot k_H + k_H = H_{\text{in}} \cdot k_H$$

## Task

Implement `transpose_conv` as a pure NumPy function for a single-channel 2D feature map. The stride is always equal to the kernel height `kH` (assume a square kernel). For each input position $(i, j)$, scatter a copy of the kernel scaled by `x[i, j]` into the corresponding non-overlapping block of the output.


In [ ]:
def transpose_conv(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Naive 2-D transposed convolution for a single-channel feature map.

    The stride is set equal to the kernel size, so adjacent output blocks
    do not overlap. This simplifies the implementation while preserving the
    key scatter-add concept.

    Parameters
    ----------
    x : np.ndarray, shape (H_in, W_in)
        Input feature map.
    w : np.ndarray, shape (kH, kW)
        Convolution kernel (square: kH == kW).

    Returns
    -------
    out : np.ndarray, shape (H_in * kH, W_in * kW)
        Transposed convolution output (non-overlapping tiling).
    """

    H_in, W_in = x.shape
    kH, kW = w.shape

    out = np.zeros((H_in * kH, W_in * kW), dtype=x.dtype)

    for i in range(H_in):
        for j in range(W_in):
            out[i*kH:(i+1)*kH, j*kW:(j+1)*kW] = x[i, j] * w

    return out

In [ ]:
# ── Task 2 Self-Test ───────────────────────────────────────────────────
# A 2×2 input with a 2×2 all-ones kernel (stride=kH=2) should tile
# each input value into a non-overlapping 2×2 block.
x_test = np.array([[2.0, 3.0], [4.0, 5.0]], dtype=np.float32)
w_test = np.ones((2, 2), dtype=np.float32)
expected = np.array([
    [2., 2., 3., 3.],
    [2., 2., 3., 3.],
    [4., 4., 5., 5.],
    [4., 4., 5., 5.],
], dtype=np.float32)
got = transpose_conv(x_test, w_test)
print("Expected output:")
print(expected)
print("\nYour output:")
print(got)
assert np.allclose(got, expected), \
    "Output does not match — check your implementation."
print("\nTask 2 self-test passed!")


In [ ]:
# Visualisation: transposed convolution upsamples a small feature map
rng_v = np.random.default_rng(7)
feat_small = rng_v.random((8, 8)).astype(np.float32)
kernel_up  = np.array([[0.50, 0.75, 0.50],
                        [0.75, 1.00, 0.75],
                        [0.50, 0.75, 0.50]], dtype=np.float32)   # bilinear-like

feat_up = transpose_conv(feat_small, kernel_up)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(feat_small, cmap="viridis", interpolation="nearest")
axes[0].set_title(f"Input feature map {feat_small.shape}")
axes[1].imshow(feat_up, cmap="viridis", interpolation="nearest")
axes[1].set_title(f"After transpose_conv (stride=kH=2) {feat_up.shape}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


---

# Part 3 — U-Net: Definition, Training & Evaluation

## Architecture

The **U-Net** was originally proposed for biomedical image segmentation ([Ronneberger et al., 2015](https://arxiv.org/abs/1505.04597)). Its characteristic shape — wide at the top, narrow in the bottleneck, wide again — gives it the "U" name.

![conv_pass](https://github.com/hpi-mlia-2026/coding_assignment_04/blob/main/images/unet_diagram.png?raw=1)

The **skip connections** (horizontal arrows) copy encoder feature maps directly to the decoder. This lets the network combine the high-level semantics from the bottleneck with the fine spatial detail from the encoder.

Since we use **SAME padding** in all convolutions, the encoder and decoder spatial sizes match exactly — no cropping needed. The model below uses `ConvTranspose2d` (the PyTorch equivalent of what you implemented in Task 1) for upsampling.


In [ ]:
def _conv_block(in_ch, out_ch):
    """Two conv layers, each followed by BatchNorm + ReLU (SAME padding)."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class SimpleUNet(nn.Module):
    """U-Net with 3 downsampling steps, BatchNorm, and transposed-conv upsampling."""

    def __init__(self, in_channels: int = 1, num_fmaps: int = 8, num_classes: int = 3):
        super().__init__()
        f = num_fmaps                             # base number of feature maps

        # ── Encoder ──────────────────────────────────────────────
        self.enc1 = _conv_block(in_channels, f)   # (1 → 8)
        self.enc2 = _conv_block(f,     f * 2)     # (8 → 16)
        self.enc3 = _conv_block(f * 2, f * 4)     # (16 → 32)

        # ── Bottleneck ───────────────────────────────────────────
        self.bottleneck = _conv_block(f * 4, f * 8)   # (32 → 64)

        # ── Decoder ──────────────────────────────────────────────
        self.up3  = nn.ConvTranspose2d(f * 8, f * 4, kernel_size=2, stride=2)
        self.dec3 = _conv_block(f * 4 + f * 4, f * 4)  # concat with enc3

        self.up2  = nn.ConvTranspose2d(f * 4, f * 2, kernel_size=2, stride=2)
        self.dec2 = _conv_block(f * 2 + f * 2, f * 2)  # concat with enc2

        self.up1  = nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2)
        self.dec1 = _conv_block(f + f, f)               # concat with enc1

        # ── Output head ──────────────────────────────────────────
        self.head = nn.Conv2d(f, num_classes, kernel_size=1)
        self.pool = nn.MaxPool2d(2)

        # Xavier weight initialisation
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        # Bottleneck
        b  = self.bottleneck(self.pool(s3))
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([s3, self.up3(b)],  dim=1))
        d2 = self.dec2(torch.cat([s2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([s1, self.up1(d2)], dim=1))
        return self.head(d1)


# Sanity-check: one forward pass through a tiny input
_dummy = torch.zeros(1, 1, 32, 32)
_out   = SimpleUNet()(_dummy)
assert _out.shape == (1, 3, 32, 32), f"Unexpected output shape: {_out.shape}"

n_params = sum(p.numel() for p in SimpleUNet().parameters())
print(f"SimpleUNet defined.  Parameters: {n_params:,}")

In [ ]:
# Instantiate the model and move to GPU
# 360×360 is divisible by 2^3 = 8, so no padding is needed.
net = SimpleUNet(in_channels=1, num_fmaps=8, num_classes=3).to(device)

loss_fn   = nn.CrossEntropyLoss()
activation = nn.Softmax(dim=1)

print("Model and loss ready.")

## Training the U-Net

We train using **AdamW** with a cosine-annealing learning-rate schedule. Validation loss is logged every 200 steps so you can monitor convergence. Training curves are also written to `./logs/` for TensorBoard (`tensorboard --logdir=./logs`).

**Expected training time on GPU: ~20 minutes** (2 000 steps, batch size 2).


In [ ]:
BATCH_SIZE    = 2
TRAINING_STEPS = 2000
LR            = 1e-3
WEIGHT_DECAY  = 1e-4

data_train = NucleusDataset(train_files, augment=True)
data_val   = NucleusDataset(val_files,   augment=False)
data_test  = NucleusDataset(test_files,  augment=False)

train_loader = DataLoader(data_train, batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(data_val,   batch_size=1,          shuffle=False, pin_memory=True)
test_loader  = DataLoader(data_test,  batch_size=1,          shuffle=False)

os.makedirs("./logs", exist_ok=True)
print(f"Train: {len(data_train)}  Val: {len(data_val)}  Test: {len(data_test)}")

In [ ]:
# Launch TensorBoard in Colab-style notebooks
log_dir = "logs"
%load_ext tensorboard
%tensorboard --logdir {log_dir}

In [ ]:
def train_unet(model, train_loader, val_loader, device,
               n_steps=2000, lr=1e-3, weight_decay=1e-4,
               log_dir="./logs", vis_img_tensor=None, vis_interval=250):
    """Step-based training loop with TensorBoard logging and periodic validation."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    writer    = SummaryWriter(log_dir=log_dir)
    history   = {"step": [], "train_loss": [], "val_loss": [], "val_acc": [],
                 "vis_steps": [], "vis_preds": []}

    def _snapshot(step):
        """Record one prediction on the fixed validation image."""
        if vis_img_tensor is None:
            return
        model.eval()
        with torch.no_grad():
            pred = model(vis_img_tensor.to(device)).argmax(1).squeeze(0).cpu().numpy()
        history["vis_steps"].append(step)
        history["vis_preds"].append(pred)
        model.train()

    _snapshot(0)   # untrained baseline
    model.train()
    step = 0
    pbar = tqdm(total=n_steps, desc="Training")

    while step < n_steps:
        for imgs, labels in train_loader:
            if step >= n_steps:
                break
            imgs   = imgs.to(device)
            labels = labels.squeeze(1).to(device)   # (B, H, W) long

            optimizer.zero_grad(set_to_none=True)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            step += 1
            pbar.update(1)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

            if step % 200 == 0:
                # ── Validation + TensorBoard ─────────────────────────────────
                model.eval()
                v_losses, v_accs = [], []
                with torch.no_grad():
                    for v_imgs, v_labels in val_loader:
                        v_imgs   = v_imgs.to(device)
                        v_labels = v_labels.squeeze(1).to(device)
                        v_logits = model(v_imgs)
                        v_losses.append(criterion(v_logits, v_labels).item())
                        v_accs.append((v_logits.argmax(1) == v_labels).float().mean().item())

                history["step"].append(step)
                history["train_loss"].append(loss.item())
                history["val_loss"].append(float(np.mean(v_losses)))
                history["val_acc"].append(float(np.mean(v_accs)))

                writer.add_scalar("Loss/train", loss.item(), step)
                writer.add_scalar("Loss/val", float(np.mean(v_losses)), step)
                writer.add_scalar("Accuracy/val", float(np.mean(v_accs)), step)

                pbar.set_postfix(
                    loss=f"{loss.item():.4f}",
                    val_acc=f"{np.mean(v_accs):.3f}",
                )
                model.train()

            if step % vis_interval == 0:
                _snapshot(step)

    pbar.close()
    writer.close()
    return history


# ── Train  ───────────────────────────
_vis_img, _ = data_val[0]
_vis_tensor = _vis_img.unsqueeze(0)   # (1, 1, H, W) — fixed validation image for progress grid

print(f"Training on {device}...")
history = train_unet(
    net, train_loader, val_loader, device,
    n_steps=TRAINING_STEPS, lr=LR, weight_decay=WEIGHT_DECAY,
    log_dir="./logs", vis_img_tensor=_vis_tensor,
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["step"], history["train_loss"], label="Train loss")
ax1.plot(history["step"], history["val_loss"],   label="Val loss")
ax1.set_xlabel("Step"); ax1.set_ylabel("CrossEntropy loss")
ax1.set_title("Loss"); ax1.legend()

ax2.plot(history["step"], history["val_acc"])
ax2.set_xlabel("Step"); ax2.set_ylabel("Pixel accuracy")
ax2.set_title("Validation pixel accuracy")

plt.suptitle("Training history", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Training progress: predicted segmentation on fixed validation image
# Snapshots at step 0 (untrained) and every 250 steps during training.
n_snaps = len(history["vis_preds"])
nrows, ncols = 3, 3   # 3×3 grid for 9 snapshots (step 0 + 8×250)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 12))
axes_flat = axes.flatten()

for i, (step_i, pred) in enumerate(zip(history["vis_steps"], history["vis_preds"])):
    axes_flat[i].imshow(NucleusDataset.label_to_rgb(pred))
    axes_flat[i].set_title(f"Step {step_i}", fontsize=10)
    axes_flat[i].axis("off")

for ax in axes_flat[n_snaps:]:
    ax.axis("off")

plt.suptitle(
    "Predicted segmentation on fixed validation image"
    " (black=BG, blue=interior, pink=boundary)",
    fontsize=11,
)
plt.tight_layout()
plt.show()


## Model Evaluation

We evaluate on the 16 held-out test images using **pixel-wise accuracy** — the fraction of pixels whose predicted class matches the ground truth. We then visualise a few test samples to get a qualitative impression.

We see that while the pixel-wise accuracy is ~96% on the test set, the model struggles mostly with the boundary class. Since this class is the least frequent, misclassifications of boundary pixels do not affect the overall accuracy.

(Optional) Feel free to experiment with different hyperparameters and see how they affect the model performance.


In [ ]:
net.eval()
all_preds, all_imgs, all_labels = [], [], []
acc_list = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze(1).to(device)
        logits = net(imgs)
        probs  = activation(logits)
        preds  = probs.argmax(dim=1)

        acc_list.append((preds == labels).float().mean().item())
        all_imgs.append(imgs.squeeze(0).squeeze(0).cpu().numpy())
        all_labels.append(labels.squeeze(0).cpu().numpy())
        all_preds.append(preds.squeeze(0).cpu().numpy())

mean_acc = float(np.mean(acc_list))
print(f"Mean test pixel accuracy: {mean_acc:.4f}  ({mean_acc*100:.1f} %)")

# Visualise 4 test samples
fig, axes = plt.subplots(4, 3, figsize=(10, 14))
for i in range(4):
    axes[i, 0].imshow(all_imgs[i],   cmap="gray")
    axes[i, 0].set_title("Raw image")
    axes[i, 1].imshow(NucleusDataset.label_to_rgb(all_labels[i]))
    axes[i, 1].set_title("Ground truth")
    axes[i, 2].imshow(NucleusDataset.label_to_rgb(all_preds[i]))
    axes[i, 2].set_title(f"Prediction  (acc={acc_list[i]:.3f})")
    for ax in axes[i]:
        ax.axis("off")

plt.suptitle("Test set predictions (black=BG, blue=interior, pink=boundary)",
             fontsize=12)
plt.tight_layout()
plt.show()

---

# Part 5 — Task 3: Intersection over Union (IoU)
<a id="task-3"></a>

## Why pixel accuracy is not enough

Pixel accuracy tells you how many pixels were classified correctly overall. However, it is misleading when classes are imbalanced — a model that always predicts background can still achieve very high accuracy if most of the image is background, while being completely useless.

**Intersection over Union (IoU)**, also known as the Jaccard index, is a much more informative metric for segmentation. For a given class it measures how much the predicted region overlaps with the true region, taking into account both false positives and false negatives. A score of 1 means perfect agreement; 0 means no overlap at all. Averaging IoU over all classes gives the **mean IoU (mIoU)**, which is the standard benchmark metric for semantic segmentation.

## Task

Implement `iou` as a pure NumPy function. Given two integer label maps and a class ID, compute the IoU for that class. Handle the edge case where the class does not appear in either mask (the result should be 1.0 — no false positives, no false negatives).


In [ ]:
def iou(pred_mask: np.ndarray, gt_mask: np.ndarray, class_id: int) -> float:
    """Compute the Intersection-over-Union for a single class.

    Parameters
    ----------
    pred_mask : np.ndarray, shape (H, W)
        Predicted class labels (integer values).
    gt_mask : np.ndarray, shape (H, W)
        Ground-truth class labels (integer values).
    class_id : int
        The class index to compute IoU for.

    Returns
    -------
    float
        IoU score in [0, 1]. Returns 1.0 if both masks have no pixels of
        this class (no false positives, no false negatives).
    """
    pred_c = (pred_mask == class_id)
    gt_c   = (gt_mask   == class_id)

    intersection = np.logical_and(pred_c, gt_c).sum()
    union        = np.logical_or(pred_c, gt_c).sum()

    if union == 0:
        return 1.0

    out = intersection / union

    return out


In [ ]:
# ── Task 3 Self-Test ───────────────────────────────────────────────────
# class 1: pred has pixels (0,0),(0,1); gt has (0,0),(0,2)
# intersection = 1 pixel, union = 3 pixels → IoU = 1/3
pred = np.array([[1, 1, 0], [0, 0, 0]], dtype=np.int64)
gt   = np.array([[1, 0, 1], [0, 0, 0]], dtype=np.int64)
expected = 1.0 / 3.0
got = iou(pred, gt, class_id=1)
print(f"Expected: {expected:.6f}")
print(f"Got:      {got:.6f}")
assert np.isclose(got, expected, atol=1e-6), \
    f"Expected IoU = 1/3 ≈ {expected:.6f}, got {got:.6f}"
print("\nTask 3 self-test passed!")

# Apply to the test set and show per-class scores
class_names = ["background", "interior", "boundary"]
print("\nIoU on test sample 0:")
for cid, cname in enumerate(class_names):
    iou_score = iou(all_preds[0], all_labels[0], class_id=cid)
    print(f"  {cname:10s}  IoU = {iou_score:.4f}")
mean_iou = np.mean([iou(all_preds[0], all_labels[0], cid) for cid in range(3)])
print(f"  mean IoU  = {mean_iou:.4f}")


We see that our model is able to distinguish background and interior pretty well (IoU almost 1), but struggles with the boundary (IoU very low).

---

# Part 6 — Instance Segmentation with CellPose

## Semantic vs. Instance Segmentation

So far the U-Net performs **semantic segmentation**: every pixel gets one of three class labels. However, two touching nuclei both labelled *interior* are indistinguishable — we cannot count them individually.

**Instance segmentation** assigns each detected object a unique integer ID. The output is a mask where 0 = background, 1 = first object, 2 = second object, etc.

## Pre-trained CellPose

[CellPose](https://www.cellpose.org/) is a generalist deep-learning model for cell and nucleus segmentation. Instead of directly predicting binary masks, it predicts a *flow field* for each pixel that points toward the cell centre. A clustering step (gradient tracking) then groups pixels into individual instances. This makes it robust to cells of varying sizes and shapes without any retraining.


In [ ]:
from cellpose import models

cellpose_model = models.CellposeModel(
    gpu=torch.cuda.is_available()
)

print("CellPose model loaded.")

In [ ]:
# CellPose expects the original image intensities, not the zero-mean/unit-variance
# version stored by NucleusDataset.  Load the raw array directly from zarr.

# Visualise
fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for i in range(4):

    # Load raw image directly from zarr
    raw_zarr = zarr.open(test_files[i])
    img_np = np.array(raw_zarr["volumes/raw"]).squeeze().astype(np.float32)

    # Run CellPose
    cp_masks, cp_flows, cp_styles = cellpose_model.eval(img_np)

    # Column 1: raw image
    axes[i, 0].imshow(img_np, cmap="gray")
    axes[i, 0].set_title(f"Raw image {i}")

    # Column 2: instance mask
    axes[i, 1].imshow(cp_masks, cmap="tab20", interpolation="nearest")
    axes[i, 1].set_title("CellPose instance mask")

    # Column 3: overlay
    axes[i, 2].imshow(img_np, cmap="gray")
    axes[i, 2].imshow(cp_masks > 0, cmap="Reds", alpha=0.45)
    axes[i, 2].set_title("Foreground overlay")

    # Remove axes
    for j in range(3):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()


---

# Part 7 — Task 4: Instance Area Statistics
<a id="task-4"></a>

Beyond counting nuclei, biologists often care about their sizes. A large spread in nucleus area can indicate cells in different stages of the cell cycle or pathological changes.

Your task is to write a function that takes a raw image and a model and returns the mean and standard deviation of nucleus areas across all detected instances, measured in pixels. Use the population standard deviation (not sample-corrected).


In [ ]:
def compute_instance_stats(image: np.ndarray) -> tuple:
    """Compute mean and standard deviation of nucleus instance areas using CellPose.
    (docstring as given)
    """
    from cellpose import models

    # CellPose v4 (Cellpose-SAM): use CellposeModel, no model_type / channels
    model = models.CellposeModel(gpu=True)

    # v4 eval returns 3 values: masks, flows, styles
    masks, flows, styles = model.eval(image)

    # Unique nonzero labels = individual nuclei
    instance_ids = np.unique(masks)
    instance_ids = instance_ids[instance_ids != 0]   # drop background

    n_inst = len(instance_ids)
    if n_inst == 0:
        return (0, 0.0, 0.0)

    # Area of each instance = pixel count for that label
    areas = np.array([(masks == i).sum() for i in instance_ids])

    mean_area = float(np.mean(areas))
    std_area  = float(np.std(areas))      # population std (ddof=0) ✓

    return (n_inst, mean_area, std_area)


In [ ]:
# ── Task 4 Self-Test ───────────────────────────────────────────────────
_raw_zarr = zarr.open(test_files[0])
demo_img_np = np.array(_raw_zarr["volumes/raw"]).squeeze().astype(np.float32)
n_inst, mean_area, std_area = compute_instance_stats(demo_img_np)
print(f"Number of instances : {n_inst:.0f} (Correct: 27)")
print(f"Mean nucleus area   : {mean_area:.1f} px² (Correct: ~1317.8)")
print(f"Std dev             : {std_area:.1f} px² (Correct: ~804.2)")



---

# Extra: Bottleneck Feature Maps

## What does the bottleneck learn?

The **bottleneck** is the narrowest point of the U-Net — the deepest layer. After three 2× pooling steps the spatial resolution is 8× smaller than the input, but the number of channels is at its maximum (64). Each feature map here encodes a high-level, abstract representation of the image content.

Visualising these feature maps gives direct insight into what the network has learned to detect.

## PyTorch forward hooks

PyTorch allows you to attach a **forward hook** to any module. The hook is a function that PyTorch calls automatically with the module's output every time a forward pass runs through that module. This lets you inspect intermediate activations without changing the model code. You register a hook with `module.register_forward_hook(...)`, which returns a handle you should call `.remove()` on afterwards to avoid memory leaks.

The function `get_bottleneck_features` below demonstrates this pattern. Run the cells to explore what the U-Net bottleneck has learned.


In [ ]:
def get_bottleneck_features(model: 'SimpleUNet', image_tensor: torch.Tensor) -> np.ndarray:
    """Extract and return the first 3 feature maps from the U-Net bottleneck.

    Parameters
    ----------
    model : SimpleUNet
        A trained SimpleUNet instance.
    image_tensor : torch.Tensor, shape (1, 1, H, W)
        Single input image (batch size = 1).

    Returns
    -------
    np.ndarray, shape (H', W', 3)
        The first three bottleneck feature maps stacked as an RGB image,
        where each channel is independently normalised to [0, 1].
        H' = H / 8, W' = W / 8  (three 2× downsampling steps).
    """
    captured = {}

    def hook_fn(module, input, output):
        captured["feats"] = output

    handle = model.bottleneck.register_forward_hook(hook_fn)
    model.eval()
    with torch.no_grad():
        _ = model(image_tensor)
    handle.remove()

    # feats: (1, C, H', W') — take first 3 channels
    feats = captured["feats"][0, :3].cpu().numpy()     # (3, H', W')

    # Normalise each channel independently to [0, 1]
    out = np.zeros((feats.shape[1], feats.shape[2], 3), dtype=np.float32)
    for ch in range(3):
        f = feats[ch]
        f_min, f_max = f.min(), f.max()
        if f_max > f_min:
            out[:, :, ch] = (f - f_min) / (f_max - f_min)
        # else: constant channel stays 0

    return out


In [ ]:
# Verify get_bottleneck_features works correctly
_tiny_net   = SimpleUNet(in_channels=1, num_fmaps=4, num_classes=3)
_tiny_input = torch.zeros(1, 1, 32, 32)
_feats = get_bottleneck_features(_tiny_net, _tiny_input)
assert isinstance(_feats, np.ndarray), f"Expected numpy array, got {type(_feats)}"
assert _feats.shape == (4, 4, 3), f"Expected shape (4, 4, 3), got {_feats.shape}"
assert _feats.min() >= -1e-6 and _feats.max() <= 1 + 1e-6
print("get_bottleneck_features verified.")

# Extract all bottleneck feature maps from the trained model via a forward hook
net.eval()
test_img_t, _ = data_test[0]
test_input = test_img_t.unsqueeze(0).to(device)   # (1, 1, 360, 360)

_captured = {}
def _hook_fn(module, input, output):
    _captured["feats"] = output

_handle = net.bottleneck.register_forward_hook(_hook_fn)
with torch.no_grad():
    _ = net(test_input)
_handle.remove()

feats_all = _captured["feats"][0].cpu().numpy()  # (64, H', W')

# Normalise each feature map independently to [0, 1]
feats_norm = np.zeros_like(feats_all)
for _c in range(feats_all.shape[0]):
    _f = feats_all[_c]
    _fmin, _fmax = _f.min(), _f.max()
    if _fmax > _fmin:
        feats_norm[_c] = (_f - _fmin) / (_fmax - _fmin)

# 5×5 grid: first cell = original image, remaining 24 = bottleneck feature maps
fig, axes = plt.subplots(5, 5, figsize=(13, 13))
axes_flat = axes.flatten()

axes_flat[0].imshow(test_img_t.squeeze().numpy(), cmap="gray")
axes_flat[0].set_title("Original", fontsize=8)
axes_flat[0].axis("off")

for _i in range(24):
    axes_flat[_i + 1].imshow(feats_norm[_i], cmap="viridis", vmin=0, vmax=1)
    axes_flat[_i + 1].set_title(f"fmap {_i}", fontsize=8)
    axes_flat[_i + 1].axis("off")

plt.suptitle(
    f"U-Net bottleneck features — original + first 24 maps"
    f"  ({feats_all.shape[1]}×{feats_all.shape[2]} px each)",
    fontsize=11,
)
plt.tight_layout()
plt.show()
